# LightGBM Regression — Predicting `duration_minutes`
### Gradient Boosted Trees with GOSS/DART, Early Stopping & SHAP Explanations

**Goal:** Build a LightGBM regressor to predict GitHub Actions workflow `duration_minutes` using YAML-derived and codebase features.

| Step | Description |
|------|-------------|
| 1 | Load & preprocess (type-cast, derive features) |
| 2 | IQR outlier removal on target |
| 3 | Median-target encode `os_label`; mean-target encode `primary_language` |
| 4 | Log1p-transform the target for better residual behaviour |
| 5 | Train/validation/test split |
| 6 | Baseline LightGBM with early stopping |
| 7 | RandomizedSearchCV for hyperparameter optimisation |
| 8 | Feature importance (split + gain) |
| 9 | Learning curves (train vs validation loss) |
| 10 | Evaluation: R², Adjusted R², MAE, RMSE, MAPE on original scale |
| 11 | Residual diagnostics |
| 12 | Export model with `joblib` |

**Why LightGBM?** Leaf-wise tree growth is faster and often more accurate than XGBoost's level-wise strategy on large datasets. GOSS sampling makes it especially efficient on skewed target distributions.

In [ ]:
# ── 0. Install & version check ───────────────────────────────────────
!pip install lightgbm==4.5.0 shap --quiet

import lightgbm as lgb
import shap
print(f"LightGBM version : {lgb.__version__}")
print(f"SHAP version     : {shap.__version__}")
print("Imports complete ✓")

In [ ]:
# ── 1. Core imports ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, os
from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint, uniform

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="darkgrid", palette="muted")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Core imports complete ✓")

In [ ]:
# ── 2. Load dataset ──────────────────────────────────────────────────
# Try multiple locations (Colab drive, local, dataset curation output)
CANDIDATE_PATHS = [
    "/content/drive/MyDrive/workflow_features.csv",
    "workflow_features.csv",
    "../GHA_dataset_curation-main (1)/GHA_dataset_curation-main/workflow_features.csv",
    "../GHA_dataset_curation-main/workflow_features.csv",
    "GHA_dataset_curation-main (1)/GHA_dataset_curation-main/workflow_features.csv",
]

df = None
for p in CANDIDATE_PATHS:
    if Path(p).exists():
        df = pd.read_csv(p)
        print(f"Loaded from: {p}")
        break

if df is None:
    raise FileNotFoundError(
        "workflow_features.csv not found. "
        "Upload it to Colab or place it in the working directory."
    )

print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# ── 3. Feature set & type casting ────────────────────────────────────
FEATURE_COLS = [
    "num_jobs", "num_steps", "num_triggers",
    "uses_cache", "uses_docker", "uses_matrix",
    "num_env_vars", "num_secrets", "num_artifacts",
    "code_complexity", "has_if_conditions",
    "parallel_jobs", "sequential_jobs", "max_matrix_size",
    "total_timeout_minutes", "uses_self_hosted",
    "os_label", "primary_language",
    # New features (present in upgraded dataset)
    "run_command_line_count", "workflow_trigger_is_pr", "repo_size_kb",
]

TARGET_COL = "duration_minutes"

# Keep only columns that exist
available_features = [c for c in FEATURE_COLS if c in df.columns]
print(f"Available features ({len(available_features)}): {available_features}")

missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f"Missing (will be skipped): {missing}")

df = df[available_features + [TARGET_COL]].copy()

# Type casting
bool_cols = ["uses_cache", "uses_docker", "uses_matrix", "has_if_conditions",
             "uses_self_hosted", "workflow_trigger_is_pr"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df.dropna(subset=[TARGET_COL], inplace=True)
df.fillna(0, inplace=True)

print(f"\nDataset shape after type casting: {df.shape}")
df.dtypes

In [ ]:
# ── 4. IQR outlier removal on target ─────────────────────────────────
Q1, Q3 = df[TARGET_COL].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR

before = len(df)
df = df[(df[TARGET_COL] >= max(0, lower)) & (df[TARGET_COL] <= upper)].copy()
print(f"Removed {before - len(df)} outliers  ({before} → {len(df)} rows)")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
df[TARGET_COL].hist(bins=60, ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title("duration_minutes (raw)")
np.log1p(df[TARGET_COL]).hist(bins=60, ax=axes[1], color='darkorange', edgecolor='none')
axes[1].set_title("log1p(duration_minutes)")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Target encoding for categorical features ───────────────────────
# Median-target encode os_label; mean-target encode primary_language
# (computed on training set only — applied to all splits below)

def fit_target_encoding(series, target, agg="median", smoothing=1.0):
    """Smoothed target encoding: blend category mean/median with global stat."""
    global_stat = target.median() if agg == "median" else target.mean()
    stats = target.groupby(series).agg(agg) if agg == "median" else target.groupby(series).mean()
    counts = series.value_counts()
    # Smoothing: w = n / (n + smoothing)
    enc = {}
    for cat in stats.index:
        n = counts.get(cat, 0)
        w = n / (n + smoothing)
        enc[cat] = w * stats[cat] + (1 - w) * global_stat
    return enc, global_stat

OS_ENC, OS_GLOBAL = None, None
LANG_ENC, LANG_GLOBAL = None, None

if "os_label" not in df.columns:
    df["os_label"] = "ubuntu-latest"
if "primary_language" not in df.columns:
    df["primary_language"] = "Unknown"

df["os_label"] = df["os_label"].astype(str).str.strip().str.lower().fillna("ubuntu-latest")
df["primary_language"] = df["primary_language"].astype(str).str.strip().str.lower().fillna("unknown")

print("os_label value counts:")
print(df["os_label"].value_counts().head(10))
print("\nprimary_language value counts:")
print(df["primary_language"].value_counts().head(10))

In [ ]:
# ── 6. Log1p-transform target ─────────────────────────────────────────
df["log_duration"] = np.log1p(df[TARGET_COL])

# ── 7. Train / validation / test split ───────────────────────────────
X = df[available_features].copy()
y = df["log_duration"].copy()
y_raw = df[TARGET_COL].copy()

X_trainval, X_test, y_trainval, y_test, y_raw_trainval, y_raw_test = train_test_split(
    X, y, y_raw, test_size=0.15, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

# Fit target encodings on TRAIN only
OS_ENC, OS_GLOBAL = fit_target_encoding(X_train["os_label"], y_train, agg="median")
LANG_ENC, LANG_GLOBAL = fit_target_encoding(X_train["primary_language"], y_train, agg="mean")

def apply_encodings(df_split):
    df_split = df_split.copy()
    df_split["os_label"] = df_split["os_label"].map(OS_ENC).fillna(OS_GLOBAL)
    df_split["primary_language"] = df_split["primary_language"].map(LANG_ENC).fillna(LANG_GLOBAL)
    return df_split

X_train = apply_encodings(X_train)
X_val = apply_encodings(X_val)
X_test = apply_encodings(X_test)

print("Target encodings applied ✓")

In [ ]:
# ── 8. Baseline LightGBM with early stopping ─────────────────────────
train_data = lgb.Dataset(X_train, label=y_train, feature_name=list(X_train.columns))
val_data   = lgb.Dataset(X_val,   label=y_val,   reference=train_data)

base_params = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "boosting_type"    : "gbdt",
    "num_leaves"       : 31,
    "learning_rate"    : 0.05,
    "feature_fraction" : 0.9,
    "bagging_fraction" : 0.8,
    "bagging_freq"     : 5,
    "verbose"          : -1,
    "random_state"     : RANDOM_STATE,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100),
]

baseline_model = lgb.train(
    base_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    callbacks=callbacks,
)

best_iter = baseline_model.best_iteration
print(f"\nBest iteration: {best_iter}")

# Evaluate on validation (log-space)
val_pred_log = baseline_model.predict(X_val, num_iteration=best_iter)
val_pred_raw = np.expm1(val_pred_log)
y_val_raw = np.expm1(y_val)

val_r2   = r2_score(y_val_raw, val_pred_raw)
val_mae  = mean_absolute_error(y_val_raw, val_pred_raw)
val_rmse = mean_squared_error(y_val_raw, val_pred_raw) ** 0.5
print(f"Validation  R²={val_r2:.4f}  MAE={val_mae:.4f}  RMSE={val_rmse:.4f}")

In [ ]:
# ── 9. Learning curves ────────────────────────────────────────────────
evals_result = {}
callbacks_lc = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.record_evaluation(evals_result),
    lgb.log_evaluation(period=-1),
]

_ = lgb.train(
    base_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    callbacks=callbacks_lc,
)

train_rmse_hist = evals_result["train"]["rmse"]
val_rmse_hist   = evals_result["val"]["rmse"]

plt.figure(figsize=(10, 4))
plt.plot(train_rmse_hist, label="Train RMSE", linewidth=1.5)
plt.plot(val_rmse_hist,   label="Val RMSE",   linewidth=1.5)
plt.axvline(best_iter - 1, color="red", linestyle="--", label=f"Best iter ({best_iter})")
plt.xlabel("Boosting Round")
plt.ylabel("RMSE (log-space)")
plt.title("LightGBM Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Hyperparameter optimisation (RandomizedSearchCV via sklearn API) ──
from lightgbm import LGBMRegressor

param_dist = {
    "n_estimators"     : randint(200, 1000),
    "num_leaves"       : randint(20, 150),
    "learning_rate"    : uniform(0.01, 0.15),
    "min_child_samples": randint(5, 50),
    "subsample"        : uniform(0.6, 0.4),
    "colsample_bytree" : uniform(0.6, 0.4),
    "reg_alpha"        : uniform(0.0, 1.0),
    "reg_lambda"       : uniform(0.0, 1.0),
    "boosting_type"    : ["gbdt", "goss"],
}

lgbm_sklearn = LGBMRegressor(
    objective="regression",
    metric="rmse",
    random_state=RANDOM_STATE,
    verbose=-1,
)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    lgbm_sklearn,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

# Fit on full train+val set for search
X_tv = pd.concat([X_train, X_val])
y_tv = pd.concat([y_train, y_val])

search.fit(X_tv, y_tv)
print("\nBest params:", search.best_params_)
print(f"Best CV RMSE (log-space): {-search.best_score_:.4f}")

In [ ]:
# ── 11. Final model with best params ─────────────────────────────────
best_params = search.best_params_.copy()
best_params.update({"objective": "regression", "metric": "rmse",
                    "verbose": -1, "random_state": RANDOM_STATE})

n_est = best_params.pop("n_estimators", 500)

train_data_tv = lgb.Dataset(X_tv, label=y_tv)
val_data_final = lgb.Dataset(X_val, label=y_val, reference=train_data_tv)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=n_est,
    valid_sets=[lgb.Dataset(X_val, label=y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1),
    ],
)

print(f"Final model best iteration: {final_model.best_iteration}")

In [ ]:
# ── 12. Test-set evaluation ───────────────────────────────────────────
def evaluate(model, X, y_log, y_original, split_name="Test"):
    pred_log = model.predict(X, num_iteration=model.best_iteration)
    pred_raw = np.expm1(pred_log)

    r2   = r2_score(y_original, pred_raw)
    n, p = len(y_original), X.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    mae   = mean_absolute_error(y_original, pred_raw)
    rmse  = mean_squared_error(y_original, pred_raw) ** 0.5
    mape  = np.mean(np.abs((y_original - pred_raw) / (y_original + 1e-9))) * 100

    print(f"\n{'='*45}")
    print(f" {split_name} Evaluation")
    print(f"{'='*45}")
    print(f"  R²           : {r2:.4f}")
    print(f"  Adjusted R²  : {adj_r2:.4f}")
    print(f"  MAE          : {mae:.4f} min")
    print(f"  RMSE         : {rmse:.4f} min")
    print(f"  MAPE         : {mape:.2f}%")
    return pred_raw

test_pred_raw = evaluate(final_model, X_test, y_test, y_raw_test, "Test")

In [ ]:
# ── 13. Feature importance ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, imp_type in zip(axes, ["split", "gain"]):
    importances = final_model.feature_importance(importance_type=imp_type)
    feat_names  = final_model.feature_name()
    feat_df = pd.DataFrame({"feature": feat_names, "importance": importances})
    feat_df = feat_df.sort_values("importance", ascending=True).tail(15)

    ax.barh(feat_df["feature"], feat_df["importance"], color="steelblue")
    ax.set_title(f"Feature Importance — {imp_type.capitalize()}")
    ax.set_xlabel(imp_type.capitalize())

plt.tight_layout()
plt.show()

In [ ]:
# ── 14. Residual diagnostics ──────────────────────────────────────────
residuals = y_raw_test.values - test_pred_raw

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Predicted vs Actual
axes[0].scatter(y_raw_test, test_pred_raw, alpha=0.3, s=10, color="steelblue")
lim = max(y_raw_test.max(), test_pred_raw.max())
axes[0].plot([0, lim], [0, lim], "r--", linewidth=1)
axes[0].set_xlabel("Actual (min)")
axes[0].set_ylabel("Predicted (min)")
axes[0].set_title("Predicted vs Actual")

# 2. Residuals vs Predicted
axes[1].scatter(test_pred_raw, residuals, alpha=0.3, s=10, color="darkorange")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted (min)")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted")

# 3. Residual distribution
axes[2].hist(residuals, bins=60, color="mediumseagreen", edgecolor="none")
axes[2].set_xlabel("Residual (min)")
axes[2].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# ── 15. SHAP explanations ─────────────────────────────────────────────
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (mean |SHAP|)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.show()

In [ ]:
# ── 16. Export model & encoding maps ─────────────────────────────────
OUTPUT_DIR = Path("../gha-cost-predictor/backend/ml_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "model_lgbm.joblib"

model_bundle = {
    "model"        : final_model,
    "model_type"   : "lightgbm",
    "feature_names": list(X_train.columns),
    "os_encoding"  : OS_ENC,
    "os_global"    : OS_GLOBAL,
    "lang_encoding": LANG_ENC,
    "lang_global"  : LANG_GLOBAL,
    "log_target"   : True,
}

joblib.dump(model_bundle, model_path)
print(f"Model bundle saved → {model_path}")

# Verify load
loaded = joblib.load(model_path)
probe  = loaded["model"].predict(X_test.head(3))
print(f"Load verification — sample predictions (log-space): {probe}")
print(f"  → original scale (expm1): {np.expm1(probe)} min")

## Summary

| Metric | Value |
|--------|-------|
| R²     | *(fill after run)* |
| Adj. R² | *(fill after run)* |
| MAE    | *(fill after run)* min |
| RMSE   | *(fill after run)* min |
| MAPE   | *(fill after run)* % |

### Key Design Decisions
- **Log1p target transform** — normalises the right-skewed distribution; model trains on log-space, predictions are back-transformed with `expm1`.
- **Smoothed target encoding** — avoids leakage; computed on training fold only; unseen categories fall back to the global statistic.
- **GOSS boosting** — one of `gbdt` / `goss` selected by hyperparameter search; GOSS is faster on large imbalanced datasets.
- **SHAP** — used for post-hoc feature importance to validate the model's learned structure.